<div align="center">
    <img src="samc.png" width="300" height="200" alt="Resized Logo">
</div>

# Proyecto Final
## Erick Samuel Martínez Calderón
## Plantel Naucalpan


Para el siguiente Proyecto Integrador se realiza un Chatbot utilizando Procesamiento de Lenguaje Natural (NLP)
En este archivo se daran los recursos y procesamiento de datos, así como las consultas al LLN 

---
Este chatbot es un programa que **responde automáticamente preguntas sobre horarios, trámites y fechas** del plantel.

### Cómo funciona en 5 pasos:

1. **Lee documentos** → Extrae información de PDFs (reglamentos, convocatorias, horarios)
2. **Organiza la información** → Divide todo en trozos pequeños (chunks) para buscar
3. **Entiende la pregunta** → Convierte lo que pregunta el usuario en números (vectores)
4. **Busca la respuesta** → Encuentra los trozos más similares a lo que preguntó
5. **Genera respuesta** → Usa LLM para escribir una respuesta clara

---


**Las herramientas que usamos:**
- **PyPDF2** → Lee archivos PDF
- **LangChain** → Divide textos largos en trozos
- **ChromaDB** → Guarda y busca información en una Base de Datos Vectorial
- **Pandas** → Lee archivos Excel/CSV (tablas)
- **Groq** → LLM para generar respuestas

In [1]:
import PyPDF2  ### Libreria para extraer texto
from langchain_text_splitters import RecursiveCharacterTextSplitter ##Clase para crear chunks
import chromadb #### Libreria para base de datos vectorial
from chromadb.utils import embedding_functions ## Incrustacioon (Embedding)
import os
import pandas as pd
##import ollama
from groq import Groq  ## api de asistente de  groq LLM



### Abre el PDF y extrae todo el texto página por página.


## ¿Qué hace esta función?

Esta función **busca un archivo PDF y extrae TODO el texto** que contiene.

**Pasos:**
1. Abre el archivo (ej: "horarios.pdf")
2. Lee página por página
3. Extrae el texto de cada página
4. Lo guarda todo junto

**Resultado:** Un texto largo con toda la información del PDF.

In [2]:
def extraer_texto_pdf(ruta_archivo):

    texto_completo = ""
    try:
        with open(ruta_archivo, 'rb') as archivo:
            lector_pdf = PyPDF2.PdfReader(archivo)
            
            for numero_pagina, pagina in enumerate(lector_pdf.pages):
                texto_pagina = pagina.extract_text()
                if texto_pagina:
                    texto_completo += texto_pagina + "\n"
                    
        return texto_completo
    except FileNotFoundError:
        return "Error: No se encontró el archivo. Revisa la ruta."

### Divide el texto largo en 'chunks' (párrafos) manteniendo el contexto.


**Cómo lo hacemos:**
- Dividimos el texto en **trozos de 350 caracteres** (≈ 50 palabras)
- Los trozos se **sobrelapan 100 caracteres** para no perder contexto

**Por qué?**
- Cada trozo es pequeño = búsqueda rápida
- Trozo se sobrepone = no pierde contexto
- Mejor precisión = encuentra lo que buscas


In [3]:
def fragmentar_texto(texto_crudo):
    separador_texto = RecursiveCharacterTextSplitter(
        chunk_size = 350,  
        chunk_overlap = 100, 
        length_function = len,
        separators=["\n\n", "\n", " ", ""] 
    )
    
    fragmentos = separador_texto.split_text(texto_crudo)
    return fragmentos

### Lee todos los archivos PDF de la carpeta para usar las funciones de extraer texto y fragmentar el texto 
### Recorriendo todos los PDF con el ciclo for

In [4]:
def cargar_carpeta_pdfs(ruta_carpeta="./documentos"):
    if not os.path.exists(ruta_carpeta):
        os.makedirs(ruta_carpeta)
        print(f"Se creó la carpeta '{ruta_carpeta}'. Pon ahí tus demás PDFs.")
        return
        
    archivos = [f for f in os.listdir(ruta_carpeta) if f.endswith('.pdf')]
    for archivo in archivos:
        ruta_completa = os.path.join(ruta_carpeta, archivo)
        texto = extraer_texto_pdf(ruta_completa)
        chunks = fragmentar_texto(texto)
        guardar_en_bd(chunks, nombre_origen=archivo)

### Lectura de archivos csv

In [5]:
def cargar_calendario_csv(ruta_csv):
    if not os.path.exists(ruta_csv):
        print(f"No se encontró el archivo CSV: {ruta_csv}")
        return

    try:
        # 'utf-8-sig' elimina automáticamente el carácter invisible
        df = pd.read_csv(ruta_csv, encoding='utf-8-sig', sep=None, engine='python')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(ruta_csv, encoding='latin-1', sep=None, engine='python')
        except Exception as e:
             print(f"Error al abrir el archivo: {e}")
             return
    except Exception as e:
        print(f"Error general al abrir el archivo: {e}")
        return
        
    # Limpiamos nombres de columnas quitando caracteres invisibles o espacios extra
    df.columns = df.columns.str.strip().str.replace('\ufeff', '')
    
    chunks_calendario = []
    
    try:
        # Extraemos los datos de las columnas ya limpias
        for _, fila in df.iterrows():
            texto_evento = f"Para el proceso de {fila['proceso']}, en las fechas {fila['fechas']}, se detalla lo siguiente: {fila['descripcion']}."
            chunks_calendario.append(texto_evento)
    except KeyError as e:
        print(f"Error de columnas: Tu archivo no tiene la columna {e}.")
        print(f"Las columnas que Python está viendo en tu archivo son: {list(df.columns)}")
        return
        
    nombre_archivo = os.path.basename(ruta_csv)
    guardar_en_bd(chunks_calendario, nombre_origen=nombre_archivo)
    print(f"Calendario '{nombre_archivo}' guardado exitosamente con {len(chunks_calendario)} eventos.")

### Crea una carpeta para la base de datos vectorial

##  ¿Qué es una "base de datos vectorial"?

Normalmente, una base de datos guarda **palabras o números**.
Una base de datos **vectorial** guarda **números que representan significado**.

**¿Cómo funciona?**

Cada trozo de texto se convierte en una lista de números:
```
"horario"      → [0.45, 0.82, 0.12, 0.91, 0.55, ...]
"calendario"   → [0.48, 0.79, 0.15, 0.88, 0.52, ...]
```

Estos números son **muy similares** = ChromaDB entiende que significan cosas parecidas.

**¿Para qué?** Para **buscar por significado**, no por palabras exactas.

**Ejemplo:**
- Usuario pregunta: "¿Cuándo empieza?"
- ChromaDB busca el **significado** de "empieza"
- Encuentra trozos sobre "inicio", "comienza", "arranca", etc.
- Responde correctamente 

In [6]:
cliente_chroma = chromadb.PersistentClient(path="./mi_base_vectorial")

In [7]:
funcion_embedding = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
coleccion_oficina = cliente_chroma.get_or_create_collection(
    name="documentos_administrativos",
    embedding_function=funcion_embedding
)

In [9]:
def guardar_en_bd(chunks_de_texto, nombre_origen):
    # Necesitamos crear un ID 
    ids = [f"{nombre_origen}_chunk_{i}" for i in range(len(chunks_de_texto))]
    
    # Metadatos opcionales para saber de dónde salió (útil si luego tienes muchos PDFs)
    metadatos = [{"origen": nombre_origen} for _ in range(len(chunks_de_texto))]
    
    # Guardar en ChromaDB
    coleccion_oficina.add(
        documents=chunks_de_texto,
        metadatas=metadatos,
        ids=ids
    )
    print(f"Se guardaron {len(chunks_de_texto)} fragmentos en la base de datos de '{nombre_origen}'.")

### Esta es la orden para arrancar el procesamiento

cargar_carpeta_pdfs("./documentos")
cargar_calendario_csv("Calendario.csv")
print("Proceso terminado.")


### Agregamos API Key para el LLM

## ¿Qué es Groq? ¿Qué es una API?

**API** = "puerta" para pedirle cosas a un servicio en internet.

**Groq** = empresa que tiene **inteligencia artificial en internet**.
Nosotros le pagamos (o usamos gratis) para que nos ayude a generar respuestas.

Es como tener un **experto en un servidor lejano** que contesta tus preguntas.

**Tu API Key** = tu "contraseña" para usar Groq.


**¿Cómo lo usamos?**
1. Le enviamos una pregunta a Groq
2. Groq procesa con su inteligencia artificial
3. Nos devuelve una respuesta escrita

In [10]:

os.environ["GROQ_API_KEY"] = "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"
cliente_groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))


In [11]:
def expandir_pregunta(pregunta_usuario):
    prompt = f"""Reescribe la siguiente pregunta de un usuario en una versión más formal y completa,
usando terminología típica de trámites administrativos escolares (CCH, Secretaría Académica, horarios, PEPASIG, PGH, etc.).
Responde ÚNICAMENTE con la pregunta reescrita, sin explicaciones ni texto adicional.

Pregunta original: {pregunta_usuario}"""

    try:
        respuesta = cliente_groq.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )
        return respuesta.choices[0].message.content.strip()
    except Exception as e:
        print(f"No se pudo expandir la pregunta, se usará la original. Detalle: {e}")
        return pregunta_usuario

In [12]:

def buscar_contexto(pregunta, n_resultados=8):
    pregunta_expandida = expandir_pregunta(pregunta)
    print(f"Pregunta expandida: {pregunta_expandida}")

    resultados_original = coleccion_oficina.query(
        query_texts=[pregunta],
        n_results=n_resultados
    )
    resultados_expandida = coleccion_oficina.query(
        query_texts=[pregunta_expandida],
        n_results=n_resultados
    )

    fragmentos_combinados = []
    vistos = set()
    for lista in [resultados_original['documents'][0], resultados_expandida['documents'][0]]:
        for doc in lista:
            if doc not in vistos:
                fragmentos_combinados.append(doc)
                vistos.add(doc)

    return "\n\n".join(fragmentos_combinados)


In [13]:

def generar_respuesta(pregunta):
    contexto = buscar_contexto(pregunta)
    prompt = f"""Eres un asistente que responde preguntas sobre trámites, horarios de la Secretaría Académica del CCH plantel Naucalpan.
Usa ÚNICAMENTE la siguiente información para responder. 
Si la información no es suficiente, dilo claramente. 
Y si no viene nada dentro de la información sugiere asistir a la Secretaría Académica y dirigirse con Samuel Martinez.  
En el caso EXCLUSIVO de que se pregunte por hablar con un humano pasar el contacto de Samuel Martínez (Cel: 5633193490, erick.samuel@cch.unam.mx)

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""

    respuesta = cliente_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
    )

    return respuesta.choices[0].message.content


### Esta celda contenía la integración con Ollama. Decarté la ejecución local debido a limitaciones de mi hardware, lo cual generaba tiempos de respuesta más largos.

'''
def preguntar_al_asistente_local(pregunta_usuario):
    # 1. Buscar en tu base vectorial ChromaDB
    resultados = coleccion_oficina.query(
        query_texts=[pregunta_usuario],
        n_results=3 
    )
    
    documentos_recuperados = resultados['documents'][0]
    
    if not documentos_recuperados:
        contexto_texto = "No se encontró información."
    else:
        contexto_texto = "\n\n".join(documentos_recuperados)
    
    # 2. Construir el prompt con los datos de tu escuela
    prompt_sistema = f"""
    Eres un asistente virtual de la Secretaría Académica. 
    Responde a la pregunta del profesor basándote ÚNICAMENTE en la siguiente información oficial.
    Usa viñetas si hay fechas o pasos. Si la información no está en el contexto, di claramente que no la encuentras.

    --- CONTEXTO OFICIAL ---
    {contexto_texto}
    ------------------------

    PREGUNTA DEL PROFESOR: {pregunta_usuario}
    """
    
    # 3. Llamada directa al Ollama local con Llama 3
    try:
        response = ollama.chat(
            model='llama3',  # Nombre exacto que sale en 'ollama list'
            messages=[
                {
                    'role': 'user', 
                    'content': prompt_sistema
                },
            ]
        )
        return response['message']['content']
        
    except Exception as e:
        return f"Error al conectar con Ollama local: Asegúrate de que Ollama esté abierto en segundo plano. Detalle: {e}"
'''

In [14]:
##print(generar_respuesta("¿Quiero hablar con un humano"))

In [15]:
##print(preguntar_al_asistente_local('¿Quien asigna los grupos?'))

### Flask

**Flask** es un framework web ligero y flexible para Python. Se clasifica como un microframework porque no requiere herramientas ni librerías particulares para funcionar, y no incluye cosas como capas de abstracción de base de datos o validación de formularios de manera predeterminada.

En lugar de imponer una estructura rígida, Flask te da lo mínimo necesario para recibir peticiones HTTP y responderlas, dejándote a ti la libertad de elegir las librerías que prefieras para el resto.

In [16]:
from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse
import threading

app_flask = Flask(__name__)

@app_flask.route("/whatsapp", methods=["POST"])
def whatsapp_webhook():
    mensaje_usuario = request.form.get("Body", "").strip()
    numero_remitente = request.form.get("From", "")
    print(f"Mensaje recibido de {numero_remitente}: {mensaje_usuario}")

    try:
        texto_respuesta = generar_respuesta(mensaje_usuario)
    except Exception as e:
        print(f"Error generando respuesta: {e}")
        texto_respuesta = "Lo siento, tuve un problema procesando tu pregunta. Intenta de nuevo."

    respuesta_twiml = MessagingResponse()
    respuesta_twiml.message(texto_respuesta)
    return str(respuesta_twiml)

@app_flask.route("/", methods=["GET"])
def home():
    return "El servidor del chatbot está funcionando correctamente."

In [17]:
def correr_flask():
    app_flask.run(port=5001)

hilo_servidor = threading.Thread(target=correr_flask, daemon=True)
hilo_servidor.start()
print("Servidor Flask corriendo en segundo plano (puerto 5001)")


Servidor Flask corriendo en segundo plano (puerto 5001)
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


In [18]:
from pyngrok import ngrok

# Solo la primera vez que uses pyngrok en esta computadora, descomenta y corre con tu authtoken:
# ngrok.set_auth_token("TU_AUTHTOKEN_AQUI")

url_publica = ngrok.connect(5001)
print(f"Tu URL pública es: {url_publica}")
print(f"Configura esto en Twilio (Sandbox settings -> When a message comes in):")
print(f"{url_publica}/whatsapp?ngrok-skip-browser-warning=true")

Tu URL pública es: NgrokTunnel: "https://headfirst-proactive-heaviness.ngrok-free.dev" -> "http://localhost:5001"
Configura esto en Twilio (Sandbox settings -> When a message comes in):
NgrokTunnel: "https://headfirst-proactive-heaviness.ngrok-free.dev" -> "http://localhost:5001"/whatsapp?ngrok-skip-browser-warning=true


In [20]:
# Detener el servidor y el túnel al terminar la demo
#ngrok.disconnect(url_publica)
#ngrok.kill()
